# BN4601A: Medical Artificial Intelligence
## Individual Assignment: Breast Cancer Classification on Mammograms (CBIS-DDSM)

**Clinical Problem Statement:**
Screening mammography is the primary imaging modality for early detection of breast cancer. However, interpreting mammograms is visually challenging due to dense fibroglandular tissue, subtle lesion margins, and radiologist fatigue.

In this project, we develop an AI Perception model using **Ultralytics YOLO11 (Image Classification)** to classify a full screening mammogram as **Benign** or **Malignant**.





In [ ]:
# os & pathlib: handle filesystem navigation and folder creation safely across Windows/Linux
import os
import random
import shutil
from pathlib import Path

# pandas: tabular data analysis (reading csv files)
import pandas as pd

# numpy: numerical array computations (finding white pixel coordinates in masks)
import numpy as np

# PIL: Python Imaging Library for opening, resizing, and drawing on images
from PIL import Image, ImageDraw

# matplotlib: plotting library to display images directly inside this notebook
import matplotlib.pyplot as plt

# Set seeds so all random splits (80/20 train/val) are 100% reproducible every time
random.seed(42)
np.random.seed(42)

print('✓ All core libraries imported successfully!')

✓ All core libraries imported successfully!


In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA version PyTorch was built with:", torch.version.cuda)

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU name: NVIDIA GeForce RTX 4050 Laptop GPU
CUDA version PyTorch was built with: 12.8


In [ ]:
# Define paths to CSV and JPEG folders
CSV_DIR = Path('csv')
JPEG_DIR = Path('jpeg')

# 1. Read the mass-case descriptions.
#    (Calcification cases were tested and rejected — see the markdown above.)
df_train_meta = pd.read_csv(CSV_DIR / 'mass_case_description_train_set.csv')
df_test_meta  = pd.read_csv(CSV_DIR / 'mass_case_description_test_set.csv')

# 2. Read dicom_info.csv which contains the mapping to disk
df_dicom = pd.read_csv(CSV_DIR / 'dicom_info.csv')

# 3. Replace Kaggle prefix 'CBIS-DDSM/jpeg/' with local folder 'jpeg/'
df_dicom['local_path'] = df_dicom['image_path'].str.replace('CBIS-DDSM/jpeg/', 'jpeg/', regex=False)

# 4. Build a fast lookup dictionary mapping (SeriesInstanceUID, SeriesDescription) -> local file path
dicom_lookup = df_dicom.set_index(['SeriesInstanceUID', 'SeriesDescription'])['local_path'].to_dict()

# Helper function: extracts the SeriesInstanceUID from the path column
def extract_series_uid(path_str):
    parts = str(path_str).strip().split('/')
    return parts[-2] if len(parts) >= 3 else None

# Helper function: converts pathology text into numeric class (0 = benign, 1 = malignant)
def map_pathology_to_class(pathology_str):
    path_clean = str(pathology_str).strip().upper()
    if 'MALIGNANT' in path_clean:
        return 1  # Malignant
    else:
        return 0  # Benign

print('Metadata successfully loaded (mass cases):')
print(f' - Train cases: {len(df_train_meta)}')
print(f' - Test cases:  {len(df_test_meta)}')
print('Class Mapping: 0 -> Benign, 1 -> Malignant')

In [ ]:
def group_cases_by_image(df_meta):
    """
    Iterates over each row in the clinical metadata,
    finds the corresponding full mammogram and ROI mask on disk,
    and groups all lesions per full image path. Also records which
    patient each image belongs to, so downstream splits can avoid
    leaking the same patient across train/val/test.
    """
    image_to_lesions = {}
    image_to_patient = {}

    for _, row in df_meta.iterrows():
        # Extract the Series IDs for full mammogram and mask
        full_uid = extract_series_uid(row['image file path'])
        roi_uid  = extract_series_uid(row['ROI mask file path'])

        # Retrieve local file paths from our lookup dictionary
        full_img_path = dicom_lookup.get((full_uid, 'full mammogram images'))
        roi_img_path  = dicom_lookup.get((roi_uid, 'ROI mask images'))

        # Ensure both files exist on disk
        if not full_img_path or not roi_img_path:
            continue
        if not os.path.exists(full_img_path) or not os.path.exists(roi_img_path):
            continue

        cls_id = map_pathology_to_class(row['pathology'])

        if full_img_path not in image_to_lesions:
            image_to_lesions[full_img_path] = []

        image_to_lesions[full_img_path].append({
            'roi_path': roi_img_path,
            'class_id': cls_id
        })
        image_to_patient[full_img_path] = row['patient_id']

    return image_to_lesions, image_to_patient

train_cases_map, train_patient_map = group_cases_by_image(df_train_meta)
test_cases_map,  test_patient_map  = group_cases_by_image(df_test_meta)

print(f'Unique full mammograms mapped:')
print(f' - Training images: {len(train_cases_map)}')
print(f' - Test images:     {len(test_cases_map)}')

In [ ]:
# Destination directory for the classification dataset
OUTPUT_DIR = Path('cls_dataset')
TARGET_SIZE = (640, 640)  # Full mammogram resolution written to disk
CLASS_NAMES = {0: 'benign', 1: 'malignant'}

# Clean slate: if this cell is re-run, delete any previous export first.
# (Without this, re-running Step 4 without a kernel restart just adds NEW
# files on top of the OLD ones under new indices, silently duplicating
# images inside train/val rather than replacing them.)
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

# Create folder structure: {train, val, test}/{benign, malignant}
for split in ['train', 'val', 'test']:
    for cname in CLASS_NAMES.values():
        (OUTPUT_DIR / split / cname).mkdir(parents=True, exist_ok=True)

# Perform an 80/20 train/validation split at the PATIENT level (not image level).
# CBIS-DDSM has multiple images per patient (left/right breast, CC/MLO views),
# so splitting by image would put the same patient in both train and val and
# inflate the validation score.
#
# We also guard against any patient appearing in both the official train and
# test CSVs. For mass-only data this is already zero, but the check is kept
# because it was essential when calcification cases were included: the mass and
# calc train/test splits were built independently per abnormality type, so the
# same patient could land in the mass "train" split AND the calc "test" split.
test_patient_ids = set(test_patient_map.values())
excluded_patients = set(train_patient_map.values()) & test_patient_ids

patient_to_images = {}
for img_path, pid in train_patient_map.items():
    if pid in excluded_patients:
        continue
    patient_to_images.setdefault(pid, []).append(img_path)

print(f'Excluded {len(excluded_patients)} patient(s) from train/val '
      f'(also present in the official test set) to prevent leakage.')

all_patient_ids = list(patient_to_images.keys())
random.shuffle(all_patient_ids)
split_idx = int(0.8 * len(all_patient_ids))
train_patient_ids = set(all_patient_ids[:split_idx])
val_patient_ids   = set(all_patient_ids[split_idx:])

split_dict = {
    'train': [p for pid in train_patient_ids for p in patient_to_images[pid]],
    'val':   [p for pid in val_patient_ids   for p in patient_to_images[pid]],
    'test':  list(test_cases_map.keys())
}

print(f'Patient-level split: {len(train_patient_ids)} train patients | {len(val_patient_ids)} val patients')

combined_cases_map = {**train_cases_map, **test_cases_map}

def image_level_label(img_path, cases_map):
    """An image is malignant if ANY of its lesions are malignant, else benign."""
    class_ids = [lesion['class_id'] for lesion in cases_map[img_path]]
    return 1 if 1 in class_ids else 0

def process_and_save_split(split_name, image_paths, cases_map, target_size=(640, 640)):
    saved_images = 0
    class_counts = {0: 0, 1: 0}

    for idx, img_path in enumerate(image_paths):
        try:
            cls_id = image_level_label(img_path, cases_map)
            cname = CLASS_NAMES[cls_id]

            full_img = Image.open(img_path).convert('RGB')
            resized_img = full_img.resize(target_size, Image.Resampling.BILINEAR)

            base_name = f'{split_name}_{idx:04d}.jpg'
            save_img_file = OUTPUT_DIR / split_name / cname / base_name
            resized_img.save(save_img_file, quality=95)

            class_counts[cls_id] += 1
            saved_images += 1

        except Exception as e:
            print(f'Error processing {img_path}: {e}')

    print(f'[{split_name.upper():5s}] Created {saved_images:4d} images '
          f'(Benign: {class_counts[0]}, Malignant: {class_counts[1]})')

# Process all splits
for split_name in ['train', 'val', 'test']:
    process_and_save_split(split_name, split_dict[split_name], combined_cases_map, TARGET_SIZE)

In [ ]:
print(f'Classification dataset root: {OUTPUT_DIR.resolve()}')
print()
for split in ['train', 'val', 'test']:
    counts = {cname: len(list((OUTPUT_DIR / split / cname).glob('*.jpg'))) for cname in CLASS_NAMES.values()}
    total = sum(counts.values())
    print(f'{split:5s}: {total:4d} images  |  benign={counts["benign"]:4d}  malignant={counts["malignant"]:4d}')

In [ ]:
def inspect_classification_labels(num_per_class=3):
    """Plots sample images from each class folder to sanity-check labels."""
    class_colors = {'benign': 'green', 'malignant': 'red'}
    samples = []
    for cname in CLASS_NAMES.values():
        imgs = sorted(list((OUTPUT_DIR / 'train' / cname).glob('*.jpg')))
        if not imgs:
            print(f'No train images found for class "{cname}". Please run Step 4 first!')
            return
        samples.extend([(p, cname) for p in random.sample(imgs, min(num_per_class, len(imgs)))])

    fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 5))
    if len(samples) == 1:
        axes = [axes]

    for ax, (img_file, cname) in zip(axes, samples):
        img = Image.open(img_file).convert('RGB')
        ax.imshow(img)
        ax.set_title(cname.capitalize(), color=class_colors[cname], fontsize=13, fontweight='bold')
        ax.axis('off')

    plt.suptitle('Sanity Check: Mammograms with Folder-Based Labels', fontsize=15, y=1.05)
    plt.tight_layout()
    plt.show()

inspect_classification_labels(3)

In [ ]:
import torch
from ultralytics import YOLO

# Retraining identical settings gave test AUC 0.603 on 16 Sep (deployed: 0.767), so evaluate the deployed checkpoint.
RETRAIN = False

if RETRAIN:
    model = YOLO('yolo11s-cls.pt')  # pretrained classification model, downloads automatically first run

    results = model.train(
        data='cls_dataset',       # root folder containing train/val/test class subfolders
        epochs=100,
        imgsz=384,                # 384 > 224: a lesion is a small fraction of a whole mammogram
        batch=32,
        device=0,                 # use the GPU
        seed=42,
        patience=20,              # stop early once validation accuracy stops improving
        workers=0,                # disable multiprocess DataLoader workers — unstable on Windows in Jupyter
        project=str(Path('runs').resolve()),   # absolute path stops Ultralytics nesting runs/classify/runs/...
        name='cbis_ddsm_cls_final',

        # --- augmentation OFF: these corrupt the diagnostic signal on grayscale X-rays ---
        auto_augment=None,        # no posterize/solarize/equalize — these fabricate lesion-like texture
        erasing=0.0,              # stop punching black boxes over the breast (and possibly the lesion)
        hsv_h=0.0,                # grayscale input — colour jitter is meaningless but still perturbs intensity
        hsv_s=0.0,
        hsv_v=0.0,

        # --- augmentation ON: geometry that preserves the diagnostic signal ---
        scale=0.2,                # crop area 80-100% (Ultralytics maps this to scale=(1.0 - 0.2, 1.0))
        fliplr=0.5,               # left/right breasts are already mirror images of each other
        flipud=0.5,               # breast tissue has no canonical up/down orientation

        # --- regularisation for a small (979-image) training set ---
        dropout=0.2,
        cos_lr=True,
    )

    BEST_WEIGHTS = str(model.trainer.best)
    print('\nBest weights saved to:', BEST_WEIGHTS)
    print('Inspect runs/cbis_ddsm_cls_final/train_batch0.jpg to confirm the augmented '
          'images are still intact, readable mammograms.')

else:
    BEST_WEIGHTS = 'space_deploy/best.pt'
    ckpt = torch.load(BEST_WEIGHTS, map_location='cpu', weights_only=False)
    train_args = ckpt['train_args']
    history = pd.DataFrame(ckpt['train_results'])

    print(f'Using deployed weights: {BEST_WEIGHTS} (trained {ckpt["date"][:10]}, Ultralytics {ckpt["version"]})')
    print('Training arguments recorded in the checkpoint:')
    for k in ['model', 'epochs', 'patience', 'batch', 'imgsz', 'seed', 'dropout', 'cos_lr',
              'auto_augment', 'erasing', 'hsv_h', 'hsv_s', 'hsv_v', 'scale', 'fliplr', 'flipud']:
        print(f'  {k:<13}= {train_args[k]}')

    best_row = history.loc[history['metrics/accuracy_top1'].idxmax()]
    best_epoch = int(best_row['epoch'])
    print(f'\nTrained for {len(history)} epochs; best validation accuracy '
          f'{best_row["metrics/accuracy_top1"]:.3f} at epoch {best_epoch} (saved as best.pt).')

    style = {
        'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb',
        'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': '#52514e', 'axes.titlecolor': '#0b0b0b',
        'xtick.color': '#898781', 'ytick.color': '#898781', 'text.color': '#52514e',
        'axes.grid': True, 'grid.color': '#e1e0d9', 'grid.linewidth': 0.6,
        'axes.spines.top': False, 'axes.spines.right': False,
    }
    TRAIN_COLOR, VAL_COLOR = '#2a78d6', '#eb6834'
    epochs = history['epoch']

    with plt.rc_context(style):
        fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(13, 4.2))

        ax_loss.plot(epochs, history['train/loss'], color=TRAIN_COLOR, lw=2, label='train')
        ax_loss.plot(epochs, history['val/loss'], color=VAL_COLOR, lw=2, label='validation')
        for col, name in [('train/loss', 'train'), ('val/loss', 'validation')]:
            ax_loss.annotate(name, (epochs.iloc[-1], history[col].iloc[-1]), xytext=(6, 0),
                             textcoords='offset points', va='center', annotation_clip=False)
        ax_loss.set_title('Loss', loc='left')
        ax_loss.set_xlabel('epoch')
        ax_loss.legend(frameon=False, loc='upper center')

        ax_acc.plot(epochs, history['metrics/accuracy_top1'], color=VAL_COLOR, lw=2)
        ax_acc.axvline(best_epoch, color='#898781', lw=1, ls='--')
        ax_acc.annotate(f'best.pt (epoch {best_epoch})', (best_epoch, ax_acc.get_ylim()[0]),
                        xytext=(-6, 8), textcoords='offset points', ha='right')
        ax_acc.set_title('Validation accuracy', loc='left')
        ax_acc.set_xlabel('epoch')

        fig.suptitle('Training history of the deployed model', x=0.01, ha='left', color='#0b0b0b')
        fig.tight_layout()
        plt.show()

In [ ]:
import numpy as np
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, classification_report
)

# Set in Step 7: the deployed checkpoint, or this session's run when RETRAIN = True.
best_model_path = globals().get('BEST_WEIGHTS', 'space_deploy/best.pt')

print('Using weights:', best_model_path)
best_model = YOLO(best_model_path)

# Map YOLO's internal class index -> name (alphabetical order: benign=0, malignant=1)
idx_to_name = best_model.names
malignant_idx = [i for i, n in idx_to_name.items() if n == 'malignant'][0]

test_dir = OUTPUT_DIR / 'test'

def score_split(split_name):
    """Return (true labels, P(malignant)) for every image in a split."""
    y_true, y_score = [], []
    for cname in CLASS_NAMES.values():
        label = 1 if cname == 'malignant' else 0
        paths = sorted((OUTPUT_DIR / split_name / cname).glob('*.jpg'))
        # Predict in batches of 32 — far faster than one call per image
        for i in range(0, len(paths), 32):
            batch = [str(p) for p in paths[i:i + 32]]
            for result in best_model.predict(source=batch, verbose=False):
                probs = result.probs.data.cpu().numpy()
                y_true.append(label)
                y_score.append(float(probs[malignant_idx]))
    return np.array(y_true), np.array(y_score)

y_val,  s_val   = score_split('val')
y_true, y_score = score_split('test')

def report(tag, y, s, thr):
    y_pred = (s >= thr).astype(int)
    print(f'\n--- {tag}  (threshold = {thr:.3f}) ---')
    print(f'Accuracy:                        {accuracy_score(y, y_pred):.3f}')
    print(f'Sensitivity (Malignant Recall):  {recall_score(y, y_pred, pos_label=1, zero_division=0):.3f}   <- most clinically important')
    print(f'Specificity (Benign Recall):     {recall_score(y, y_pred, pos_label=0, zero_division=0):.3f}')
    print(f'Precision (Malignant):           {precision_score(y, y_pred, pos_label=1, zero_division=0):.3f}')
    print(f'F1 (Malignant):                  {f1_score(y, y_pred, pos_label=1, zero_division=0):.3f}')
    print(f'AUC:                             {roc_auc_score(y, s):.3f}')
    print('Confusion Matrix [rows=true, cols=pred] (0=benign, 1=malignant):')
    print(confusion_matrix(y, y_pred))
    return y_pred

print(f'\nValidation set: {len(y_val)} images | Test set: {len(y_true)} images')

# Operating point 1: the default argmax threshold
y_pred = report('TEST @ default', y_true, y_score, 0.5)
print()
print(classification_report(y_true, y_pred, target_names=['benign', 'malignant']))

# Operating point 2: lowest threshold on the VALIDATION ROC reaching the target
# sensitivity, then applied unchanged to the test set (never fitted on test).
TARGET_SENSITIVITY = 0.85
fpr, tpr, thresholds = roc_curve(y_val, s_val)
reaches_target = np.where(tpr >= TARGET_SENSITIVITY)[0]
TUNED_THRESHOLD = float(np.clip(thresholds[reaches_target[0]], 0.0, 1.0)) if len(reaches_target) else 0.5

print(f'\nThreshold chosen on VALIDATION for sensitivity >= {TARGET_SENSITIVITY:.2f}: {TUNED_THRESHOLD:.3f}')
report(f'TEST @ val-tuned for sensitivity >= {TARGET_SENSITIVITY:.2f}', y_true, y_score, TUNED_THRESHOLD)

In [ ]:
def show_predictions(num_correct=3, num_wrong=3):
    """Displays a mix of correct and incorrect predictions on unseen test images."""
    test_paths = []
    for cname in CLASS_NAMES.values():
        true_label = 1 if cname == 'malignant' else 0
        for img_path in sorted((test_dir / cname).glob('*.jpg')):
            test_paths.append((img_path, true_label))

    correct, wrong = [], []
    for img_path, true_label in test_paths:
        result = best_model.predict(source=str(img_path), verbose=False)[0]
        probs = result.probs.data.cpu().numpy()
        pred_idx = int(probs.argmax())
        pred_label = 1 if idx_to_name[pred_idx] == 'malignant' else 0
        conf = float(probs[pred_idx])
        entry = (img_path, true_label, pred_label, conf)
        (correct if pred_label == true_label else wrong).append(entry)

    random.shuffle(correct)
    random.shuffle(wrong)
    samples = correct[:num_correct] + wrong[:num_wrong]

    if not samples:
        print('No predictions to show.')
        return

    fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 5))
    if len(samples) == 1:
        axes = [axes]

    for ax, (img_path, true_label, pred_label, conf) in zip(axes, samples):
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
        is_ok = (true_label == pred_label)
        title = (f"Pred: {CLASS_NAMES[pred_label]} ({conf:.0%})\nTrue: {CLASS_NAMES[true_label]}")
        ax.set_title(title, color=('green' if is_ok else 'red'), fontsize=11, fontweight='bold')
        ax.axis('off')

    plt.suptitle('Predictions on Held-Out Test Images', fontsize=15, y=1.05)
    plt.tight_layout()
    plt.show()

show_predictions(3, 3)

In [ ]:
import gradio as gr

# Operating point for the demo's triage decision. Step 8 selects this on the
# VALIDATION ROC (never on test); this fallback keeps the cell runnable if
# Step 8 hasn't been executed in this kernel session.
DEMO_THRESHOLD = globals().get('TUNED_THRESHOLD', 0.5)

def classify_mammogram(image):
    results = best_model.predict(source=image, verbose=False)
    probs = results[0].probs.data.cpu().numpy()
    scores = {idx_to_name[i]: float(probs[i]) for i in range(len(probs))}

    p_malignant = float(probs[malignant_idx])
    if p_malignant >= DEMO_THRESHOLD:
        decision = (
            f'### ⚠️ FLAG FOR REVIEW\n'
            f'P(malignant) = **{p_malignant:.1%}**, at or above the {DEMO_THRESHOLD:.0%} '
            f'referral threshold.'
        )
    else:
        decision = (
            f'### ✅ No flag raised\n'
            f'P(malignant) = **{p_malignant:.1%}**, below the {DEMO_THRESHOLD:.0%} '
            f'referral threshold.'
        )
    decision += (
        f'\n\nThe threshold is deliberately set below 50% so the system errs toward '
        f'over-referral: in screening, a false alarm costs a recall appointment, '
        f'whereas a missed cancer costs a delayed diagnosis.'
    )
    return scores, decision

example_imgs = (
    [str(p) for p in list((OUTPUT_DIR / 'test' / 'benign').glob('*.jpg'))[:2]] +
    [str(p) for p in list((OUTPUT_DIR / 'test' / 'malignant').glob('*.jpg'))[:2]]
)

demo = gr.Interface(
    fn=classify_mammogram,
    inputs=gr.Image(type='pil', label='Upload Screening Mammogram'),
    outputs=[
        gr.Label(num_top_classes=2, label='Class Probabilities'),
        gr.Markdown(label='Triage Decision'),
    ],
    title='AI Perception System: Breast Cancer Classification on Mammography',
    description=(
        'Upload a full screening mammogram to estimate Benign vs. Malignant. '
        'Educational demonstration only — not for diagnosis or clinical decision-making. (BN4601A)'
    ),
    examples=example_imgs
)

demo.launch(share=False)